# Tomato Leaf Disease Detection - Backbone Experiment and Validation

This notebook documents the first project stage: backbone experimentation, dataset inspection, training/validation setup, evaluation, and the reasoning that led to the final EfficientNet-based pipeline.

**Portfolio note:** This notebook is part of a cleaned public portfolio version. Large datasets, trained model weights, HPC runtime artifacts, and private machine paths are not included.

**Public repository sections:**

- Project Context
- Dataset Overview
- Backbone Model Experiment
- Training and Validation Setup
- Evaluation Metrics
- Key Findings
- Next Steps Toward Final Pipeline


# Track B Experiment 1 — Presentation Notebook

This notebook is a cleaned presentation version of **Experiment 1**. It reuses the already-saved Track B results for model selection, builds presentation visuals, exports a full dataset Excel catalog, and does **not** retrain the models.

## Section 1 — Paths And Presentation Output Setup

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('.')
DATA_ROOT = PROJECT_ROOT / 'Universal_Tomato_Dataset'
SOURCE_RESULTS_ROOT = PROJECT_ROOT / 'trackB_outputs' / 'Track_B_Experiment_1'
PRESENTATION_ROOT = PROJECT_ROOT / 'trackB_outputs' / 'Track_B_Experiment_1_Presentation'

SAVE_DIRS = {
    'root': PRESENTATION_ROOT,
    'figures': PRESENTATION_ROOT / 'figures',
    'tables': PRESENTATION_ROOT / 'tables',
    'excel': PRESENTATION_ROOT / 'excel',
    'galleries': PRESENTATION_ROOT / 'galleries',
    'reports': PRESENTATION_ROOT / 'reports',
    'configs': PRESENTATION_ROOT / 'configs',
}

for path in SAVE_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

required_paths = {
    'project_root': PROJECT_ROOT,
    'data_root': DATA_ROOT,
    'source_results_root': SOURCE_RESULTS_ROOT,
}
missing_paths = [name for name, path in required_paths.items() if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Missing required paths: {missing_paths}')

print('Project root         :', PROJECT_ROOT)
print('Dataset root         :', DATA_ROOT)
print('Source results root  :', SOURCE_RESULTS_ROOT)
print('Presentation outputs :', PRESENTATION_ROOT)


## Section 2 — Imports, Style, And Helper Functions

In [ ]:
import json
import random
import textwrap
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
sns.set_theme(style='whitegrid')

PRESENTATION_COLORS = {
    'navy': '#12355B',
    'teal': '#1B998B',
    'gold': '#F4A259',
    'red': '#BC4749',
    'green': '#4C956C',
    'gray': '#6C757D',
    'light': '#F7FAFC',
}

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10


def read_json(path: Path) -> Dict:
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def load_rgb_image(path: str) -> np.ndarray:
    with Image.open(path) as img:
        return np.array(img.convert('RGB'))


def human_readable_int(x: int) -> str:
    return f'{int(x):,}'


def display_label(name: str) -> str:
    return str(name).replace('_', ' ')


def save_styled_table_figure(
    df: pd.DataFrame,
    path: Path,
    title: str,
    figsize: Tuple[int, int] = (14, 6),
    font_size: int = 10,
) -> None:
    table_df = df.copy().fillna('').astype(str)

    max_col_lengths = {
        col: max([len(str(col))] + table_df[col].map(len).tolist())
        for col in table_df.columns
    }
    total_col_length = max(sum(max_col_lengths.values()), 1)
    total_wrap_budget = max(20 * len(table_df.columns), 84)

    wrap_widths = {
        col: max(
            16,
            min(40, round(total_wrap_budget * max_col_lengths[col] / total_col_length)),
        )
        for col in table_df.columns
    }

    wrapped_df = table_df.copy()
    for col in wrapped_df.columns:
        width = wrap_widths[col]
        wrapped_df[col] = wrapped_df[col].map(
            lambda x, width=width: textwrap.fill(
                x,
                width=width,
                break_long_words=False,
                break_on_hyphens=False,
            )
        )

    wrapped_columns = [
        textwrap.fill(str(col), width=max(14, wrap_widths[col]))
        for col in wrapped_df.columns
    ]

    row_line_counts = wrapped_df.apply(
        lambda row: max(max(1, len(str(value).splitlines())) for value in row),
        axis=1,
    )

    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    ax.set_title(title, pad=16, fontsize=16, fontweight='bold')

    col_widths = np.array(
        [
            max(max_col_lengths[col], np.mean(list(max_col_lengths.values())) * 0.65)
            for col in wrapped_df.columns
        ],
        dtype=float,
    )
    col_widths = (col_widths / col_widths.sum()).tolist()

    table = ax.table(
        cellText=wrapped_df.values,
        colLabels=wrapped_columns,
        cellLoc='left',
        colLoc='left',
        colWidths=col_widths,
        bbox=[0, 0, 1, 0.86],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(font_size)

    header_weight = 1.2
    row_weights = [max(1.0, 0.9 + 0.45 * (line_count - 1)) for line_count in row_line_counts]
    total_height_weight = header_weight + sum(row_weights)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor('#25364A')
        cell.set_linewidth(0.9)
        cell.PAD = 0.10
        cell.get_text().set_ha('left')
        cell.get_text().set_va('center')

        if row == 0:
            cell.set_facecolor(PRESENTATION_COLORS['navy'])
            cell.set_text_props(color='white', weight='bold')
            cell.set_height(0.86 * header_weight / total_height_weight)
        else:
            cell.set_height(0.86 * row_weights[row - 1] / total_height_weight)
            cell.set_facecolor('white' if row % 2 == 1 else PRESENTATION_COLORS['light'])

    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches='tight', facecolor='white')
    plt.show()

## Section 3 — Load Saved Experiment 1 Results

In [ ]:
PHASE1_EXPERIMENTS = [
    'b1_tf_efficientnet_b3_224',
    'b1_tf_efficientnet_b4_224',
    'b1_convnext_tiny_224',
]
PHASE2_EXPERIMENTS = [
    'b2_tf_efficientnet_b3_224',
    'b2_tf_efficientnet_b3_300',
    'b2_tf_efficientnet_b3_384',
]
PHASE3_EXPERIMENTS = [
    'b3_384_tf_efficientnet_b3_lr_0.0003',
    'b3_384_tf_efficientnet_b3_lr_0.0001',
    'b3_384_tf_efficientnet_b3_lr_5e-05',
]

MODEL_NAME_MAP = {
    'tf_efficientnet_b3': 'EfficientNet-B3',
    'tf_efficientnet_b4': 'EfficientNet-B4',
    'convnext_tiny': 'ConvNeXt-Tiny',
}


def load_experiment_summary(exp_name: str) -> Dict:
    summary_path = SOURCE_RESULTS_ROOT / exp_name / 'summary.json'
    if not summary_path.exists():
        raise FileNotFoundError(f'Missing summary file: {summary_path}')
    summary = read_json(summary_path)
    summary['summary_path'] = str(summary_path)
    summary['history_csv'] = str(SOURCE_RESULTS_ROOT / exp_name / f'{exp_name}_history.csv')
    return summary


def build_phase_df(exp_names: List[str], phase_label: str) -> pd.DataFrame:
    rows = []
    for exp_name in exp_names:
        summary = load_experiment_summary(exp_name)
        row = {
            'phase': phase_label,
            'exp_name': exp_name,
            'best_epoch': summary['best_epoch'],
            'val_acc_pct': round(summary['final_val_acc'] * 100, 2),
            'val_macro_f1_pct': round(summary['final_val_macro_f1'] * 100, 2),
            'test_acc_pct': round(summary['final_test_acc'] * 100, 2),
            'test_macro_f1_pct': round(summary['final_test_macro_f1'] * 100, 2),
            'aug_mode': summary['aug_mode'],
        }

        if phase_label == 'Phase 1':
            row['display_label'] = MODEL_NAME_MAP[exp_name.replace('b1_', '').replace('_224', '')]
            row['x_label'] = row['display_label']
        elif phase_label == 'Phase 2':
            img_size = int(exp_name.rsplit('_', 1)[-1])
            row['img_size'] = img_size
            row['display_label'] = f'{img_size}x{img_size}'
            row['x_label'] = row['display_label']
        elif phase_label == 'Phase 3':
            lr_label = exp_name.split('_lr_')[-1]
            row['learning_rate'] = lr_label
            row['display_label'] = lr_label
            row['x_label'] = lr_label
        rows.append(row)
    return pd.DataFrame(rows)

phase1_df = build_phase_df(PHASE1_EXPERIMENTS, 'Phase 1').sort_values('val_macro_f1_pct', ascending=False).reset_index(drop=True)
phase2_df = build_phase_df(PHASE2_EXPERIMENTS, 'Phase 2').sort_values('img_size').reset_index(drop=True)
phase3_df = build_phase_df(PHASE3_EXPERIMENTS, 'Phase 3').reset_index(drop=True)
all_phase_df = pd.concat([phase1_df, phase2_df, phase3_df], ignore_index=True)
all_phase_df.to_csv(SAVE_DIRS['tables'] / 'experiment1_all_runs_summary.csv', index=False)

overall_winner_row = phase1_df.sort_values('val_macro_f1_pct', ascending=False).iloc[0]
best_resolution_row = phase2_df.sort_values('val_macro_f1_pct', ascending=False).iloc[0]
best_lr_row = phase3_df.sort_values('val_macro_f1_pct', ascending=False).iloc[0]

print('Experiment 1 presentation notebook is ready.')
print('Selected backbone family :', overall_winner_row['display_label'])
print('Selection metric         : Validation Macro-F1')
print('Winning value            :', f"{overall_winner_row['val_macro_f1_pct']:.2f}%")
print('Best image size          :', best_resolution_row['display_label'])
print('Best learning rate       :', best_lr_row['display_label'])
print('Presentation output root :', PRESENTATION_ROOT)


## Section 4 — Dataset Catalog And Excel Export

In [ ]:
def count_images_in_folder(folder: Path) -> int:
    return sum(
        1 for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def scan_dataset_to_inventory(dataset_root: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    inventory_rows = []
    folder_rows = []

    for split_dir in sorted([p for p in dataset_root.iterdir() if p.is_dir()]):
        split = split_dir.name

        for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()], key=lambda x: x.name.lower()):
            class_name = class_dir.name
            image_count = count_images_in_folder(class_dir)

            folder_rows.append({
                'split': split,
                'class_name': class_name,
                'folder_path': str(class_dir),
                'image_count': image_count,
            })

            for image_path in sorted(class_dir.iterdir(), key=lambda x: x.name.lower()):
                if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue

                try:
                    with Image.open(image_path) as img:
                        width, height = img.size
                        mode = img.mode
                        fmt = img.format
                except Exception:
                    width, height, mode, fmt = np.nan, np.nan, 'UNREADABLE', 'UNKNOWN'

                inventory_rows.append({
                    'split': split,
                    'class_name': class_name,
                    'file_name': image_path.name,
                    'file_path': str(image_path),
                    'relative_path': str(image_path.relative_to(dataset_root)),
                    'extension': image_path.suffix.lower(),
                    'width': width,
                    'height': height,
                    'aspect_ratio': (width / height) if pd.notna(width) and pd.notna(height) and height else np.nan,
                    'megapixels': ((width * height) / 1_000_000) if pd.notna(width) and pd.notna(height) else np.nan,
                    'file_size_kb': image_path.stat().st_size / 1024,
                    'mode': mode,
                    'format': fmt,
                })

    return pd.DataFrame(inventory_rows), pd.DataFrame(folder_rows)


inventory_df, folder_structure_df = scan_dataset_to_inventory(DATA_ROOT)

split_order = ['train', 'val', 'test']
inventory_df['split'] = pd.Categorical(inventory_df['split'], categories=split_order, ordered=True)
folder_structure_df['split'] = pd.Categorical(folder_structure_df['split'], categories=split_order, ordered=True)


dataset_overview_df = pd.DataFrame([
    {
        'dataset_root': str(DATA_ROOT),
        'total_images': int(len(inventory_df)),
        'num_classes': int(inventory_df['class_name'].nunique()),
        'num_splits': int(inventory_df['split'].nunique()),
        'avg_width': round(float(inventory_df['width'].mean()), 2),
        'avg_height': round(float(inventory_df['height'].mean()), 2),
        'avg_megapixels': round(float(inventory_df['megapixels'].mean()), 3),
    }
])

split_summary_df = (
    inventory_df.groupby('split', observed=False)
    .size()
    .reset_index(name='image_count')
    .sort_values('split')
    .reset_index(drop=True)
)

split_class_counts_df = (
    inventory_df.groupby(['split', 'class_name'], observed=False)
    .size()
    .reset_index(name='image_count')
    .sort_values(['split', 'class_name'])
    .reset_index(drop=True)
)

class_summary_df = (
    inventory_df.groupby('class_name')
    .agg(
        total_images=('file_path', 'count'),
        avg_width=('width', 'mean'),
        avg_height=('height', 'mean'),
        avg_megapixels=('megapixels', 'mean'),
    )
    .reset_index()
    .sort_values('total_images', ascending=False)
    .reset_index(drop=True)
)

resolution_summary_df = (
    inventory_df.groupby('split', observed=False)
    .agg(
        image_count=('file_path', 'count'),
        mean_width=('width', 'mean'),
        median_width=('width', 'median'),
        mean_height=('height', 'mean'),
        median_height=('height', 'median'),
        mean_megapixels=('megapixels', 'mean'),
        mean_file_size_kb=('file_size_kb', 'mean'),
    )
    .round(2)
    .reset_index()
)

excel_path = SAVE_DIRS['excel'] / 'dataset_catalog.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    dataset_overview_df.to_excel(writer, sheet_name='dataset_overview', index=False)
    split_summary_df.to_excel(writer, sheet_name='split_summary', index=False)
    split_class_counts_df.to_excel(writer, sheet_name='split_class_counts', index=False)
    resolution_summary_df.to_excel(writer, sheet_name='resolution_summary', index=False)
    class_summary_df.to_excel(writer, sheet_name='class_summary', index=False)
    folder_structure_df.to_excel(writer, sheet_name='folder_structure', index=False)
    inventory_df.to_excel(writer, sheet_name='file_inventory', index=False)

inventory_df.to_csv(SAVE_DIRS['excel'] / 'file_inventory.csv', index=False)
split_class_counts_df.to_csv(SAVE_DIRS['tables'] / 'split_class_counts.csv', index=False)
class_summary_df.to_csv(SAVE_DIRS['tables'] / 'class_summary.csv', index=False)
folder_structure_df.to_csv(SAVE_DIRS['tables'] / 'folder_structure.csv', index=False)

print(f'Dataset catalog saved to: {excel_path}')
print('Total images             :', human_readable_int(len(inventory_df)))
print('Number of classes        :', inventory_df['class_name'].nunique())
print('Split counts             :', ', '.join(f"{row.split}={human_readable_int(row.image_count)}" for row in split_summary_df.itertuples()))


## Section 5 — Data Source And Preprocessing Visuals

In [ ]:
# Dataset Split Overview
plt.figure(figsize=(8.5, 5.2))
ax = sns.barplot(
    data=split_summary_df,
    x='split',
    y='image_count',
    hue='split',
    dodge=False,
    palette=[PRESENTATION_COLORS['teal'], PRESENTATION_COLORS['gold'], PRESENTATION_COLORS['red']],
    legend=False,
)
plt.title('Dataset Split Overview')
plt.xlabel('Split')
plt.ylabel('Number of Images')

for patch in ax.patches:
    value = int(patch.get_height())
    ax.annotate(
        human_readable_int(value),
        (patch.get_x() + patch.get_width() / 2, value),
        ha='center',
        va='bottom',
        fontsize=10,
        xytext=(0, 5),
        textcoords='offset points',
    )

plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'dataset_split_overview.png', dpi=220)
plt.show()

# Image Resolution Statistics: Width vs Height
plt.figure(figsize=(9.5, 7.2))
hb = plt.hexbin(
    inventory_df['width'],
    inventory_df['height'],
    gridsize=45,
    cmap='viridis',
    mincnt=1,
)
cb = plt.colorbar(hb)
cb.set_label('Image Count')
plt.title('Image Resolution Statistics: Width vs Height')
plt.xlabel('Width (pixels)')
plt.ylabel('Height (pixels)')
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'image_resolution_width_vs_height.png', dpi=220)
plt.show()

# Extra visual: class balance heatmap across splits
heatmap_df = (
    split_class_counts_df
    .pivot(index='class_name', columns='split', values='image_count')
    .fillna(0)
)
heatmap_df = heatmap_df.reindex(columns=['train', 'val', 'test'])
heatmap_df.index = [display_label(name) for name in heatmap_df.index]

plt.figure(figsize=(11, 8.5))
sns.heatmap(heatmap_df, annot=True, fmt='.0f', cmap='YlGnBu', linewidths=0.4)
plt.title('Class Balance Heatmap Across Train / Val / Test')
plt.xlabel('Split')
plt.ylabel('Disease Class')
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'class_balance_heatmap.png', dpi=220)
plt.show()


## Section 6 — Augmentation Policy Used In Experiment 1

In [ ]:
augmentation_plan_df = pd.DataFrame(
    [
        {
            'Stage': 'Train',
            'Augmentation': 'Resize',
            'Settings': 'Resize to each test image size',
            'Purpose': 'Keeps the input setup fair across models.',
        },
        {
            'Stage': 'Train',
            'Augmentation': 'HorizontalFlip',
            'Settings': 'p = 0.50',
            'Purpose': 'Adds left-right variation.',
        },
        {
            'Stage': 'Train',
            'Augmentation': 'Affine',
            'Settings': 'light scale / shift / rotate / shear',
            'Purpose': 'Adds gentle viewpoint variation.',
        },
        {
            'Stage': 'Train',
            'Augmentation': 'Brightness / Contrast',
            'Settings': 'light intensity adjustment',
            'Purpose': 'Adds mild lighting variation.',
        },
        {
            'Stage': 'Train',
            'Augmentation': 'GaussianBlur',
            'Settings': 'blur_limit=(3, 3), p = 0.08',
            'Purpose': 'Adds slight camera softness.',
        },
        {
            'Stage': 'Train + Eval',
            'Augmentation': 'ImageNet Normalization',
            'Settings': 'mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)',
            'Purpose': 'Matches pretrained backbone statistics.',
        },
        {
            'Stage': 'Validation / Test',
            'Augmentation': 'Resize + Normalize Only',
            'Settings': 'No random augmentation',
            'Purpose': 'Keeps validation and test evaluation consistent.',
        },
    ]
)

augmentation_plan_df.to_csv(SAVE_DIRS['tables'] / 'augmentation_plan_experiment1.csv', index=False)
save_styled_table_figure(
    augmentation_plan_df,
    SAVE_DIRS['figures'] / 'augmentation_plan_table.png',
    title='Augmentation Strategy Used In Experiment 1',
    figsize=(16.5, 4.8),
    font_size=10,
)

print('Augmentation table saved to:', SAVE_DIRS['figures'] / 'augmentation_plan_table.png')


## Section 7 — Disease Gallery Table With 7 Images Per Class

In [ ]:
gallery_source_df = inventory_df[inventory_df['split'] == 'train'].copy()
gallery_samples = []
random.seed(42)

for class_name in sorted(gallery_source_df['class_name'].unique()):
    class_df = gallery_source_df[gallery_source_df['class_name'] == class_name].copy()
    sample_df = class_df.sample(n=min(7, len(class_df)), random_state=42)
    sample_df['sample_rank'] = range(1, len(sample_df) + 1)
    gallery_samples.append(sample_df)

gallery_samples_df = pd.concat(gallery_samples, ignore_index=True)
gallery_samples_df.to_csv(SAVE_DIRS['galleries'] / 'gallery_samples_used.csv', index=False)

class_order = sorted(gallery_samples_df['class_name'].unique())
n_rows = len(class_order)
n_cols = 8

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(18, 1.9 * n_rows + 1.0),
    squeeze=False,
    gridspec_kw={
        'width_ratios': [1.35] + [1] * 7,
        'wspace': 0.03,
        'hspace': 0.08,
    },
)

for row_idx, class_name in enumerate(class_order):
    row_df = (
        gallery_samples_df[gallery_samples_df['class_name'] == class_name]
        .sort_values('sample_rank')
        .reset_index(drop=True)
    )

    label_ax = axes[row_idx, 0]
    label_ax.axis('off')
    label_ax.text(
        0.98,
        0.5,
        display_label(class_name),
        fontsize=10,
        weight='bold',
        va='center',
        ha='right',
        wrap=True,
    )

    for col_idx in range(7):
        ax = axes[row_idx, col_idx + 1]
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)

        if col_idx < len(row_df):
            image_path = row_df.iloc[col_idx]['file_path']
            with Image.open(image_path) as img:
                ax.imshow(img.convert('RGB'))
            ax.set_aspect('equal')
            ax.set_anchor('C')

        if row_idx == 0:
            ax.set_title(f'Sample {col_idx + 1}', fontsize=9, weight='bold', pad=3)

fig.suptitle(
    'Disease Gallery Table: 7 Example Images Per Class',
    fontsize=15,
    weight='bold',
    y=0.995,
)

fig.subplots_adjust(
    left=0.01,
    right=0.995,
    top=0.975,
    bottom=0.01,
    wspace=0.03,
    hspace=0.08,
)

plt.savefig(
    SAVE_DIRS['galleries'] / 'disease_gallery_table.png',
    dpi=220,
    bbox_inches='tight',
    pad_inches=0.03,
    facecolor='white',
)
plt.show()


## Section 8 — Experiment 1 Design And Selection Journey

In [ ]:
selection_journey_df = pd.DataFrame(
    [
        {
            'Phase': 'Phase 1',
            'Goal': 'Choose the strongest backbone under one shared setup.',
            'Compared Options': 'EfficientNet-B3, EfficientNet-B4, ConvNeXt-Tiny',
            'Main Takeaway': 'EfficientNet-B3 gave the best validation macro-F1.',
        },
        {
            'Phase': 'Phase 2',
            'Goal': 'Check how image size changes EfficientNet-B3 performance.',
            'Compared Options': '224x224, 300x300, 384x384',
            'Main Takeaway': 'Higher resolutions improved results.',
        },
        {
            'Phase': 'Phase 3',
            'Goal': 'Check how learning rate changes EfficientNet-B3 performance.',
            'Compared Options': '3e-4, 1e-4, 5e-5',
            'Main Takeaway': 'Lower learning rates gave the best balance.',
        },
        {
            'Phase': 'Final Outcome',
            'Goal': 'Carry the winner into the final pipeline work.',
            'Compared Options': 'Winner from the earlier phases',
            'Main Takeaway': 'Experiment 1 selected EfficientNet-B3 for later pipeline improvement.',
        },
    ]
)

selection_journey_df.to_csv(SAVE_DIRS['tables'] / 'experiment1_selection_journey.csv', index=False)
save_styled_table_figure(
    selection_journey_df,
    SAVE_DIRS['figures'] / 'experiment1_selection_journey.png',
    title='Experiment 1: Model Selection Journey',
    figsize=(18, 6),
    font_size=10,
)

print('Selection journey table saved to:', SAVE_DIRS['figures'] / 'experiment1_selection_journey.png')


## Section 9 — Phase 1 Backbone Comparison Visuals

In [ ]:
phase1_display_df = phase1_df[[
    'display_label',
    'best_epoch',
    'val_macro_f1_pct',
    'test_acc_pct',
    'test_macro_f1_pct',
]].rename(
    columns={
        'display_label': 'Backbone',
        'best_epoch': 'Best Epoch',
        'val_macro_f1_pct': 'Val Macro-F1 (%)',
        'test_acc_pct': 'Test Accuracy (%)',
        'test_macro_f1_pct': 'Test Macro-F1 (%)',
    }
)
phase1_display_df.to_csv(SAVE_DIRS['tables'] / 'phase1_backbone_summary.csv', index=False)

save_styled_table_figure(
    phase1_display_df,
    SAVE_DIRS['figures'] / 'phase1_backbone_summary_table.png',
    title='Phase 1 Backbone Comparison Summary',
    figsize=(15, 4.2),
    font_size=10,
)

x = np.arange(len(phase1_df))
width = 0.24

plt.figure(figsize=(12.5, 6.3))
ax = plt.gca()

bars1 = ax.bar(
    x - width,
    phase1_df['val_macro_f1_pct'],
    width=width,
    color=PRESENTATION_COLORS['teal'],
    label='Val Macro-F1',
    edgecolor='white',
    linewidth=1.2,
)
bars2 = ax.bar(
    x,
    phase1_df['test_acc_pct'],
    width=width,
    color=PRESENTATION_COLORS['gold'],
    label='Test Accuracy',
    edgecolor='white',
    linewidth=1.2,
)
bars3 = ax.bar(
    x + width,
    phase1_df['test_macro_f1_pct'],
    width=width,
    color=PRESENTATION_COLORS['red'],
    label='Test Macro-F1',
    edgecolor='white',
    linewidth=1.2,
)

ax.set_xticks(x)
ax.set_xticklabels(phase1_df['display_label'], rotation=10)
ax.set_ylabel('Score (%)')
ax.set_xlabel('Backbone')
ax.set_ylim(84, 100)
ax.set_title('Phase 1: Backbone Comparison Using Standard Metrics', pad=14)
ax.grid(axis='y', linestyle='--', alpha=0.28)
ax.set_axisbelow(True)
ax.legend(loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.15,
            f'{height:.2f}',
            ha='center',
            va='bottom',
            fontsize=8,
            rotation=90,
        )

plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'phase1_backbone_comparison.png', dpi=220, bbox_inches='tight')
plt.show()

# Validation macro-F1 learning curves from saved histories
plt.figure(figsize=(11.5, 6))
for exp_name, label, color in zip(
    PHASE1_EXPERIMENTS,
    [MODEL_NAME_MAP['tf_efficientnet_b3'], MODEL_NAME_MAP['tf_efficientnet_b4'], MODEL_NAME_MAP['convnext_tiny']],
    [PRESENTATION_COLORS['teal'], PRESENTATION_COLORS['gold'], PRESENTATION_COLORS['red']],
):
    history_df = pd.read_csv(SOURCE_RESULTS_ROOT / exp_name / f'{exp_name}_history.csv')
    best_idx = history_df['val_macro_f1'].idxmax()
    plt.plot(history_df['epoch'], history_df['val_macro_f1'] * 100, label=label, linewidth=2.2, color=color, alpha=0.95)
    plt.scatter(
        history_df.loc[best_idx, 'epoch'],
        history_df.loc[best_idx, 'val_macro_f1'] * 100,
        color=color,
        s=48,
        zorder=3,
    )

plt.title('Phase 1: Validation Macro-F1 Across Training')
plt.xlabel('Epoch')
plt.ylabel('Validation Macro-F1 (%)')
plt.grid(alpha=0.22, linestyle='--')
plt.ylim(88.5, 97.2)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'phase1_validation_curves.png', dpi=220)
plt.show()


## Section 10 — Phase 2 Resolution Study Visuals

In [ ]:
phase2_display_df = phase2_df[[
    'display_label',
    'best_epoch',
    'val_macro_f1_pct',
    'test_acc_pct',
    'test_macro_f1_pct',
]].rename(
    columns={
        'display_label': 'Input Size',
        'best_epoch': 'Best Epoch',
        'val_macro_f1_pct': 'Val Macro-F1 (%)',
        'test_acc_pct': 'Test Accuracy (%)',
        'test_macro_f1_pct': 'Test Macro-F1 (%)',
    }
)
phase2_display_df.to_csv(SAVE_DIRS['tables'] / 'phase2_resolution_summary.csv', index=False)

save_styled_table_figure(
    phase2_display_df,
    SAVE_DIRS['figures'] / 'phase2_resolution_summary_table.png',
    title='Phase 2 Resolution Study Summary',
    figsize=(14, 4.2),
    font_size=10,
)

plt.figure(figsize=(10.5, 5.8))
plt.plot(phase2_df['img_size'], phase2_df['val_macro_f1_pct'], marker='o', linewidth=2.5, color=PRESENTATION_COLORS['teal'], label='Val Macro-F1')
plt.plot(phase2_df['img_size'], phase2_df['test_macro_f1_pct'], marker='o', linewidth=2.5, color=PRESENTATION_COLORS['red'], label='Test Macro-F1')
plt.plot(phase2_df['img_size'], phase2_df['test_acc_pct'], marker='o', linewidth=2.5, color=PRESENTATION_COLORS['gold'], label='Test Accuracy')

for _, row in phase2_df.iterrows():
    plt.text(row['img_size'], row['val_macro_f1_pct'] + 0.15, f"{row['val_macro_f1_pct']:.2f}", ha='center', fontsize=9)

plt.title('Phase 2: How Input Resolution Changed Performance')
plt.xlabel('Input Resolution')
plt.ylabel('Score (%)')
plt.xticks(phase2_df['img_size'], [f'{x}x{x}' for x in phase2_df['img_size']])
plt.ylim(89, 98)
plt.grid(alpha=0.28)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'phase2_resolution_comparison.png', dpi=220)
plt.show()


## Section 11 — Phase 3 Learning-Rate Study Visuals

In [ ]:
phase3_display_df = phase3_df[[
    'display_label',
    'best_epoch',
    'val_macro_f1_pct',
    'test_acc_pct',
    'test_macro_f1_pct',
]].rename(
    columns={
        'display_label': 'Learning Rate',
        'best_epoch': 'Best Epoch',
        'val_macro_f1_pct': 'Val Macro-F1 (%)',
        'test_acc_pct': 'Test Accuracy (%)',
        'test_macro_f1_pct': 'Test Macro-F1 (%)',
    }
)
phase3_display_df.to_csv(SAVE_DIRS['tables'] / 'phase3_lr_summary.csv', index=False)

save_styled_table_figure(
    phase3_display_df,
    SAVE_DIRS['figures'] / 'phase3_lr_summary_table.png',
    title='Phase 3 Learning-Rate Study Summary',
    figsize=(14, 4.2),
    font_size=10,
)

plot_lr_df = phase3_df.copy()
plot_lr_df['lr_sort'] = plot_lr_df['learning_rate'].astype(float)
plot_lr_df = plot_lr_df.sort_values('lr_sort').reset_index(drop=True)

plt.figure(figsize=(10.5, 5.8))
ax = plt.gca()
bar = ax.bar(
    plot_lr_df['display_label'],
    plot_lr_df['val_macro_f1_pct'],
    color=PRESENTATION_COLORS['teal'],
    edgecolor='white',
    linewidth=1.2,
)
ax.plot(plot_lr_df['display_label'], plot_lr_df['test_macro_f1_pct'], color=PRESENTATION_COLORS['red'], marker='o', linewidth=2.2, label='Test Macro-F1')
ax.plot(plot_lr_df['display_label'], plot_lr_df['test_acc_pct'], color=PRESENTATION_COLORS['gold'], marker='o', linewidth=2.2, label='Test Accuracy')

for patch, value in zip(bar, plot_lr_df['val_macro_f1_pct']):
    ax.text(
        patch.get_x() + patch.get_width() / 2,
        patch.get_height() + 0.12,
        f'{value:.2f}',
        ha='center',
        va='bottom',
        fontsize=9,
    )

ax.set_title('Phase 3: Learning-Rate Comparison For EfficientNet-B3')
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Score (%)')
ax.set_ylim(89, 99)
ax.grid(axis='y', linestyle='--', alpha=0.28)
ax.set_axisbelow(True)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(SAVE_DIRS['figures'] / 'phase3_learning_rate_comparison.png', dpi=220)
plt.show()


## Section 12 — Final Experiment 1 Winner And Presentation Summary

In [ ]:
experiment1_summary_df = pd.DataFrame(
    [
        {
            'Conclusion': 'Winning backbone family',
            'Evidence': 'EfficientNet-B3 delivered the strongest Phase 1 validation macro-F1.',
        },
        {
            'Conclusion': 'Why Experiment 1 mattered',
            'Evidence': 'It gave a clear model-selection path before the final pipeline work.',
        },
        {
            'Conclusion': 'How it connects to later work',
            'Evidence': 'Experiment 3 later used this winning EfficientNet-B3 backbone in the stronger final pipeline.',
        },
    ]
)

experiment1_summary_df.to_csv(SAVE_DIRS['tables'] / 'experiment1_final_summary.csv', index=False)
save_styled_table_figure(
    experiment1_summary_df,
    SAVE_DIRS['figures'] / 'experiment1_final_summary_table.png',
    title='Experiment 1 Final Takeaway',
    figsize=(16, 3.8),
    font_size=10,
)

print('Experiment 1 final takeaway table saved to:', SAVE_DIRS['figures'] / 'experiment1_final_summary_table.png')
